#Drive Mounting

In [1]:
from huggingface_hub import login
import os
from google.colab import drive

drive.mount('/content/drive')

os.environ["TRANSFORMERS_CACHE"] = "/content/drive/Shareddrives/Algoverse_KSAC/hf_cache" #stores model
os.environ["HF_HOME"] = "/content/drive/Shareddrives/Algoverse_KSAC/hf_home"  # stores logins

# hf_token = YOUR_KEY_HERE

login()

Mounted at /content/drive


#Variable Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
file_path = "/content/drive/Shareddrives/Algoverse_KSAC/shared_variable.pkl"

import pickle

with open(file_path, 'rb') as file:
    loaded_variable = pickle.load(file)
print(loaded_variable)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
0     What work, tradition or theory does Spaceballs...
1     What work, tradition or theory does Bullet Tra...
2     What work, tradition or theory does Q Who refe...
3     What work, tradition or theory does The Tin Dr...
4     What work, tradition or theory does Back to th...
5     What is the dominant foot or preferred stance ...
6     What is the dominant foot or preferred stance ...
7     What is the dominant foot or preferred stance ...
8     What is the dominant foot or preferred stance ...
9     Where was Ahwak recorded, Olympic Studios or A...
10    Where was The Snow Queen recorded, Õru or Tall...
11    Where was The Lost Treasure for Aquila recorde...
12    Where was Hinatazaka de Aimashō recorded, Tele...
13    Where was Soundtrack recorded, Fullerton Colle...
14    What is the tempo marking for Beauty and the B...
15    What is the tempo marking

list

In [ ]:
import pandas as pd

translated_data = pd.DataFrame(loaded_variable)


translated_data.head()

,SAE output
0,"What work, tradition or theory does Spaceballs..."
1,"What work, tradition or theory does Bullet Tra..."
2,"What work, tradition or theory does Q Who refe..."
3,"What work, tradition or theory does The Tin Dr..."
4,"What work, tradition or theory does Back to th..."


In [ ]:
!nvidia-smi

Sat Oct  4 05:37:16 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             29W /   70W |   12674MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#Retrieving Entities + Relations with FALCON

In [ ]:
import requests


#Getting entities for datasets

In [ ]:
# url= "https://labs.tib.eu/falcon/falcon2/api?mode=long"

# def get_entities(translated_data):
#   answers = []
#   for i in range(len(translated_data)):
#     question = translated_data.iloc[i,0]
#     data = {f'text': question}
#     headers = {"Content-Type": "application/json"}
#     answer = requests.post(url,json=data, headers=headers)
#     answers.append(answer.json())
#   return answers







#Getting entities for single questions

In [ ]:
url= "https://labs.tib.eu/falcon/falcon2/api?mode=long"

def get_entities(question):
    data = {f'text': question}
    headers = {"Content-Type": "application/json"}
    answer = requests.post(url,json=data, headers=headers)
    return answer.json()


In [ ]:
question = ("What is the first book published in the Lord of the Rings book trilogy?")

In [ ]:
answers = get_entities(question)

In [ ]:
print(answers)

{'entities_wikidata': [{'URI': 'http://www.wikidata.org/entity/Q190214', 'surface form': 'Lord of the Rings book trilogy'}, {'URI': 'http://www.wikidata.org/entity/Q56431177', 'surface form': 'book'}], 'relations_wikidata': [{'URI': 'http://www.wikidata.org/entity/P577', 'surface form': 'published'}]}


In [ ]:
def extract_entities(answers):
    entity_ids = []
    relation_ids = []
    # Entity extraction
    for e in answers.get('entities_wikidata', []):
        uri = e.get('URI', '')
        if uri.startswith('http://www.wikidata.org/entity/'):
            entity_ids.append(uri.split('/')[-1]) #Getting the entity name and storing it in a list

    # Extract relations
    for r in answers.get('relations_wikidata', []):
        uri = r.get('URI', '')
        if uri.startswith('http://www.wikidata.org/entity/'):
            relation_ids.append(uri.split('/')[-1])

    return {
        'entities': entity_ids,
        'relations': relation_ids
    }

results = extract_entities(answers)

print(results)

{'entities': ['Q190214', 'Q56431177'], 'relations': ['P577']}


In [ ]:
str(results)

"{'entities': ['Q190214', 'Q56431177'], 'relations': ['P577']}"

In [ ]:
def entity_as_l(results):
 entities = results.get("entities")
 return entities

entity = entity_as_l(results)
print(entity)

['Q190214', 'Q56431177']


In [ ]:
def relation_as_l(results):
  relation = (results.get("relations"))
  return relation

relation = relation_as_l(results)

#Text to SPARQl with QWEN

In [ ]:
# !pip install bitsandbytes accelerate

In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 43.5 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


save_path = "/content/drive/Shareddrives/Algoverse_KSAC/hf_models/Qwen/Qwen3-8B"

evaltok = AutoTokenizer.from_pretrained(save_path, local_files_only=True, enable_thinking = False)

# bNb_config = BitsAndBytesConfig( #4-bit quant
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype="float16",
#     bnb_4bit_use_double_quant=True,

# )


evalmodel = AutoModelForCausalLM.from_pretrained(
    save_path,
    local_files_only=True,
    dtype='auto',
    device_map="auto",
    # quantization_config = bNb_config
)

print("Reloaded model successfully")
print(f"model.device = {evalmodel.device}") #Verifying device

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: could not determine the shape of object type 'torch.storage.UntypedStorage'

In [ ]:
!nvidia-smi

Sat Oct  4 05:37:24 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             29W /   70W |   12674MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
def evaluate(question, entity, relation):
    messages = [
        {
            "role": "system",
            "content": (
   """ You are a SPARQL query constructor for Wikidata.

Your task is to generate a syntactically correct and executable SPARQL query given the provided question, entities (QIDs), and relations (PIDs).

Rules:
1. Output only a single SPARQL query string. Treat it as one continuous string, not a list or array.
2. Do NOT include any explanations, comments, markdown formatting, or metadata.
3. Do NOT add label services (e.g., SERVICE wikibase:label) or prefixes.
4. Do NOT use commas or string formatting symbols (such as quotes or brackets) inside the output.
5. Do NOT wrap your output in parentheses or quotation marks.
6. The output must always begin in the form:
   SELECT ?variable WHERE { ... } where variable is something contextually relevant to the question
7. Generate only the minimal triple patterns necessary for the given entities and relations.
8. add the wd: clause in front of queries and the wdt: clause in front of relations
9. The direction of the properties must be semantically correct.
10. For the property P179 ("part of the series")
    - If the given entity represents a specific work reverse the direction so that the query retrieves all items that share the same series.

Your output must be executable as-is on the Wikidata SPARQL endpoint."""




            )
        },
        {
            "role": "user",
            "content": f"question: {question}, entity: {entity}, relation: {relation}"
        }
    ]

    inputs = evaltok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking = False
    ).to(evalmodel.device)

    outputs = evalmodel.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        temperature=0.1,
        eos_token_id=evaltok.eos_token_id,

    )

    output_answer = evaltok.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],  # Slice off input tokens to get only the generated continuation
        skip_special_tokens=True
    ).split('\n')

    return output_answer


query = evaluate(question,entity, relation)
print(query)

['SELECT ?book WHERE { wd:Q190214 wdt:P577 ?publicationDate . ?book wdt:P577 ?publicationDate . ?book wdt:P179 wd:Q190214 . }']


In [ ]:
def clean(query):
    cleaned = [line.replace(",", "") for line in query]  # remove commas
    return " ".join(cleaned).strip()  # join into one string and trim spaces


query = clean(query).strip()


In [ ]:
print(query)

SELECT ?location WHERE { wd:Q127050 wdt:P625 ?location }


#Retrieval from the wikidata API

In [ ]:

USER_AGENT = "Colab-SAETOWikidata-SPARQL/1.0 (contact: saketsan8@gmail.com)"
WD_SEARCH_API = "https://www.wikidata.org/w/api.php"
WDQS_ENDPOINT = "https://query.wikidata.org/sparql"
import json





def run_sparql(query, user_agent=USER_AGENT):
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": user_agent
    }
    r = requests.get(WDQS_ENDPOINT, params={"query": query}, headers=headers, timeout=60)
    r.raise_for_status()
    data = r.json()
    simplified = []
    # Loop through each result in "bindings"
    for binding in data.get("results", {}).get("bindings", []):
        clean_entry = {}
        for key, val in binding.items():
          value = val.get("value", "")
          clean_entry[key] = value

        if clean_entry:
            simplified.append(clean_entry)
    return simplified
# Example SPARQL query — find Einstein's wives



QUERY = query
# Run and print the clean structured result
result = run_sparql(QUERY)

context = json.dumps(result, indent=2, ensure_ascii=False)

print(context)



[]


In [ ]:
links = []
for d in result:
  for v in d.values():
    links.append(v)

In [ ]:


def get_name(link, user_agent=USER_AGENT):
    entity_id = link.split("/")[-1].strip()
    url = WD_SEARCH_API
    params = {
        "action": "wbgetentities",
        "ids": entity_id,
        "format": "json",
        "languages": "en"
    }
    headers = {"User-Agent": user_agent}
    r = requests.get(url, params=params, headers=headers)
    data = r.json()
    return data["entities"][entity_id]["labels"]["en"]["value"]
for l in links:
  print(get_name(l))



In [ ]:
result

result_named = [{k:get_name(v)} for d in result for k,v in d.items()]

In [ ]:
result_named

[]